# Notebook 01 — Tracking Pipeline

Runs the full tracking pipeline without stitching.
OC-SORT track IDs are used directly — `stitched_id` equals `orig_id`.

## Stages
1. Setup & configuration
2. (Optional) Background subtraction
3. Draw vial ROIs
4. RF-DETR + OC-SORT tracking → wide CSV
5. Passthrough (no stitching) → stitched long CSV with stitched_id == orig_id
6. Vial assignment + compact IDs → compact_tracks.csv
7. Overlay video rendering

**Replace all `PLACEHOLDER` paths with your actual file paths.**

In [11]:
import sys
sys.path.insert(0, '..')

import json
import os
import re
import cv2
import yaml
import pandas as pd
from pathlib import Path
from IPython.display import Video

from src.preprocessing import preprocess_bgsub_gui
from src.metrics import run_diagnostics, compute_stitching_objectives, print_stitching_objectives
from src.tracking import export_tracks_xy_tuple_csv_one_config
from src.stitching import wide_to_long
from src.roi import draw_and_save_vial_rois, assign_vials_and_compact_ids
from src.visualization import render_vial_overlay_video, render_raw_overlay_video, render_detections_video
from utils import save_run_params

## 1 — Configuration

Set your paths and Roboflow credentials here.

In [12]:
# ---- EDIT THESE ----
RAW_VIDEO = r"C:\Users\emmav\Downloads\superfly\outputs\run_80_13DPE_n002\2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_002-converted.mp4"
MODEL_ID  = "flies-123/2"

# Load API key from creds_config.yaml (not committed to git)
with open("../creds_config.yaml", "r") as f:
    creds_config = yaml.safe_load(f)
API_KEY = creds_config["API_KEY"]

# Load defaults from config.yaml (override below if needed)
with open("../config.yaml") as _f:
    _cfg = yaml.safe_load(_f)
_t = _cfg.get("tracker", {})
_s = _cfg.get("stitching", {})
_p = _cfg.get("preprocessing", {})
_rf = _cfg.get("roboflow", {})

inference_api_url           = _rf.get("inference_api_url", "https://detect.roboflow.com")
detection_confidence_rfdetr = _t.get("detection_confidence_rfdetr", 0.4)
confidence              = _t.get("confidence", 0.1)
lost_track_buffer       = _t.get("lost_track_buffer", 90)
min_matching_threshold  = _t.get("minimum_matching_threshold", 0.2)
min_consecutive_frames  = _t.get("minimum_consecutive_frames", 3)
asso_func               = _t.get("asso_func", "diou")
brownian_pos_noise      = _t.get("brownian_pos_noise", 1.0)
aspect_weight           = _t.get("aspect_weight", 0.05)
behavioral_weight       = _t.get("behavioral_weight", 0.05)
jump_factor             = _t.get("jump_factor", 2.0)
jump_iou_threshold      = _t.get("jump_iou_threshold", 0.05)
jump_inertia            = _t.get("jump_inertia", 0.05)
bg_gain                 = _p.get("bg_gain", 1.2)
bg_white_level          = _p.get("bg_white_level", 245)
bg_percentile           = _p.get("bg_percentile", 85.0)
bg_sample_stride        = _p.get("bg_sample_stride", 1)

# Extract short label from the "N DPE/NNN" directory convention in the video path.
_m = re.search(r'(\d+)\s+DPE[/\\](\d+)', RAW_VIDEO)
short_name = f"{_m.group(1)}DPE_n{_m.group(2).zfill(3)}" if _m else Path(RAW_VIDEO).stem[:20]

# Auto-increment output directory
_outputs_root = Path("../outputs")
_outputs_root.mkdir(parents=True, exist_ok=True)
_existing = [d for d in _outputs_root.iterdir() if d.is_dir() and d.name.startswith("run_")]
_next_n = max((int(d.name.split("_")[1]) for d in _existing if d.name.split("_")[1].isdigit()), default=0) + 1
_dir_name = f"run_{_next_n}_{_m.group(1)}DPE_n{_m.group(2).zfill(3)}" if _m else f"run_{_next_n}"
OUTPUT_PATH = str(_outputs_root / _dir_name)

os.makedirs(OUTPUT_PATH, exist_ok=True)

import shutil
_dest_video = os.path.join(OUTPUT_PATH, Path(RAW_VIDEO).name)
if not os.path.exists(_dest_video):
    try:
        os.link(RAW_VIDEO, _dest_video)
    except OSError:
        shutil.copy2(RAW_VIDEO, _dest_video)
PATH_TO_VID = RAW_VIDEO

print("Output dir:", OUTPUT_PATH)
print("Short name:", short_name)
print(f"inference_api_url={inference_api_url}")
print(f"detection_confidence_rfdetr={detection_confidence_rfdetr}, asso_func={asso_func}")
print(f"aspect_weight={aspect_weight}, behavioral_weight={behavioral_weight}")
print(f"jump_factor={jump_factor}, jump_iou_threshold={jump_iou_threshold}, jump_inertia={jump_inertia}")
_cap = cv2.VideoCapture(RAW_VIDEO)
save_run_params(OUTPUT_PATH, "config", {
    "video": RAW_VIDEO, "output_dir": OUTPUT_PATH, "short_name": short_name,
    "video_fps": _cap.get(cv2.CAP_PROP_FPS),
    "video_width": int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "video_height": int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    "video_frames": int(_cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    "tracker": {"detection_confidence_rfdetr": detection_confidence_rfdetr,
                 "confidence": confidence, "lost_track_buffer": lost_track_buffer,
                 "min_matching_threshold": min_matching_threshold,
                 "min_consecutive_frames": min_consecutive_frames, "asso_func": asso_func,
                 "brownian_pos_noise": brownian_pos_noise,
                 "aspect_weight": aspect_weight, "behavioral_weight": behavioral_weight,
                 "jump_factor": jump_factor, "jump_iou_threshold": jump_iou_threshold,
                 "jump_inertia": jump_inertia},
    "preprocessing": {"bg_gain": bg_gain, "bg_white_level": bg_white_level,
                       "bg_percentile": bg_percentile, "bg_sample_stride": bg_sample_stride},
})
_cap.release()

Output dir: ..\outputs\run_107
Short name: 2024-02-12_NEG-008_h
inference_api_url=https://serverless.roboflow.com
detection_confidence_rfdetr=0.4, asso_func=diou
aspect_weight=0.05, behavioral_weight=0.05
jump_factor=2.0, jump_iou_threshold=0.05, jump_inertia=0.05


## 2 — (Optional) Background subtraction + temporal trim

Opens a GUI to:
1. Draw a **crop ROI** (spatial — pixels outside it are removed).
2. Pick a **start/end frame range** (temporal — frames outside `[start, end)` are *discarded*; only frames inside are kept).

The output `_pp.mp4` is shorter than the source: spatially cropped, temporally trimmed, and with the **85th-percentile** background subtracted. All downstream steps (tracking, stitching, overlays) run on this trimmed clip.

In [13]:
ROI_LIBRARY = Path("../roi_library.json")
_video_key = Path(RAW_VIDEO).stem

# Load existing library (or start fresh)
if ROI_LIBRARY.exists():
    with open(ROI_LIBRARY) as f:
        _library = json.load(f)
else:
    _library = {}

_crop_params = None
RAW_CROPPED_VIDEO = None
_use_saved_roi = _cfg.get("roi", {}).get("use_saved_roi", True)
preprocess = True  # set to False to skip bg subtraction entirely

if preprocess:
    pp_out = os.path.join(OUTPUT_PATH, Path(RAW_VIDEO).stem + "_pp.mp4")
    raw_cropped_out = os.path.join(OUTPUT_PATH, Path(RAW_VIDEO).stem + "_raw_cropped.mp4")
    _stored_crop = _library.get(_video_key, {}).get("preprocessing") if _use_saved_roi else None

    if _use_saved_roi and _stored_crop is not None:
        print(f"Found stored preprocessing params for: {_video_key}")
    else:
        if not _use_saved_roi:
            print("use_saved_roi=False - opening preprocessing GUI...")
        else:
            print(f"No stored preprocessing params for: {_video_key} - opening GUI...")

    pp_path, _crop_params = preprocess_bgsub_gui(
        video_path=RAW_VIDEO,
        out_mp4=pp_out,
        out_raw_mp4=raw_cropped_out,
        gain=bg_gain,
        white_level=bg_white_level,
        bg_sample_stride=bg_sample_stride,
        bg_percentile=bg_percentile,
        crop_params=_stored_crop if _use_saved_roi else None,
    )
    PATH_TO_VID = Path(pp_path)
    RAW_CROPPED_VIDEO = Path(raw_cropped_out)

    # Save crop params + full video path to library
    if _video_key not in _library:
        _library[_video_key] = {}
    _library[_video_key]["preprocessing"] = _crop_params
    _library[_video_key]["video_path"] = RAW_VIDEO
    ROI_LIBRARY.parent.mkdir(parents=True, exist_ok=True)
    with open(ROI_LIBRARY, "w") as f:
        json.dump(_library, f, indent=2)
    print("Preprocessing params saved to library.")

    # Save crop_roi.json to this run folder (allows skipping GUI on re-runs)
    with open(os.path.join(OUTPUT_PATH, "crop_roi.json"), "w") as _f:
        json.dump(_crop_params, _f, indent=2)

save_run_params(OUTPUT_PATH, "preprocessing",
                {"video_pp": str(PATH_TO_VID), "video_raw_cropped": str(RAW_CROPPED_VIDEO) if RAW_CROPPED_VIDEO is not None else None, "crop_params": _crop_params})


Found stored preprocessing params for: 2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_002-converted
Using stored crop params: x=209, y=157, w=745, h=426, frames=0–333
Saved raw cropped video: ..\outputs\run_107\2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_002-converted_raw_cropped.mp4
Saved bgsub video: ..\outputs\run_107\2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_002-converted_pp.mp4
Background (85.0th percentile) from 324 frames (stride=1).
Preprocessing params saved to library.


## 3 — Draw vial ROIs

Opens an OpenCV GUI on frame 0: drag rectangles around each vial.
Press **q** when all 6 ROIs are drawn. Saved to `vial_rois.json`.

This is a one-time step — reuse the JSON for the same experimental setup.

In [14]:
ROI_JSON = os.path.join(OUTPUT_PATH, "vial_rois.json")
_use_saved_roi = _cfg.get("roi", {}).get("use_saved_roi", True)
_stored_vials = _library.get(_video_key, {}).get("vial_rois")

if _use_saved_roi and _stored_vials is not None:
    print(f"Found stored vial ROIs for: {_video_key}")
    _vials = {k: tuple(v) for k, v in _stored_vials.items()}
    with open(ROI_JSON, "w") as f:
        json.dump({k: list(v) for k, v in _vials.items()}, f, indent=2)
    print(f"Loaded {len(_vials)} vials from library.")
else:
    if not _use_saved_roi:
        print("use_saved_roi=False — opening GUI...")
    else:
        print(f"No stored vial ROIs for: {_video_key} — opening GUI...")
    _vials = draw_and_save_vial_rois(video_path=RAW_VIDEO, roi_json_path=ROI_JSON)

    # Save to library
    if _video_key not in _library:
        _library[_video_key] = {}
    _library[_video_key]["vial_rois"] = {k: list(v) for k, v in _vials.items()}
    ROI_LIBRARY.parent.mkdir(parents=True, exist_ok=True)
    with open(ROI_LIBRARY, "w") as f:
        json.dump(_library, f, indent=2)
    print("Vial ROIs saved to library.")

save_run_params(OUTPUT_PATH, "roi", {k: list(v) for k, v in _vials.items()})

Found stored vial ROIs for: 2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_002-converted
Loaded 6 vials from library.


## 4 — RF-DETR + OC-SORT tracking

Runs the detector + tracker on every frame and writes a wide CSV.
This is the most time-consuming step. 

In [15]:
WIDE_CSV    = os.path.join(OUTPUT_PATH, "tracks_wide_format.csv")
DET_LOG_CSV = os.path.join(OUTPUT_PATH, "detections_raw.csv")

# ── Detection cache ───────────────────────────────────────────────────────────
# Set CACHED_DETS to a previous run's detections_raw.csv to skip RF-DETR.
# Leave as None to run inference and save fresh detections to DET_LOG_CSV.
CACHED_DETS = r"C:\Users\emmav\Downloads\superfly\outputs\run_80_13DPE_n002\detections_raw.csv"

_det_source = CACHED_DETS if (CACHED_DETS and os.path.exists(CACHED_DETS)) else DET_LOG_CSV
if CACHED_DETS and os.path.exists(CACHED_DETS):
    print(f"Using cached detections: {CACHED_DETS}")
else:
    print("No cache found — running RF-DETR inference")

df_wide, tracker, df_relinked = export_tracks_xy_tuple_csv_one_config(
    video_path=str(PATH_TO_VID),
    output_csv=WIDE_CSV,
    api_key=API_KEY,
    model_id=MODEL_ID,
    inference_api_url=inference_api_url,
    detection_confidence_rfdetr=detection_confidence_rfdetr,
    confidence=confidence,
    lost_track_buffer=lost_track_buffer,
    minimum_matching_threshold=min_matching_threshold,
    minimum_consecutive_frames=min_consecutive_frames,
    asso_func=asso_func,
    brownian_pos_noise=brownian_pos_noise,
    det_log_csv=_det_source,
    vial_rois=_vials,
    aspect_weight=aspect_weight,
    behavioral_weight=behavioral_weight,
    jump_factor=jump_factor,
    jump_iou_threshold=jump_iou_threshold,
    jump_inertia=jump_inertia,
    max_frames=None,
    relinked_csv=os.path.join(OUTPUT_PATH, "tracks_relinked.csv"),
)

print(df_wide.shape)
save_run_params(OUTPUT_PATH, "tracker_output", {
    "wide_csv": WIDE_CSV, "frames": int(df_wide.shape[0]), "track_count": int(df_wide.shape[1] - 1),
})
df_wide.head()

with open(os.path.join(OUTPUT_PATH, "tracker_log.json"), "w") as _f:
    json.dump({
        "detection_log":     tracker.detection_log,
        "suppressed_tracks": tracker.suppressed_tracks,
        "min_hits":          tracker.min_hits,
        "max_age":           tracker.max_age,
    }, _f)

render_detections_video(
    video_path=str(PATH_TO_VID),
    det_log_csv=_det_source,
    out_mp4=os.path.join(OUTPUT_PATH, f"{short_name}_detections_RF-DETR.mp4"),
)

Using cached detections: C:\Users\emmav\Downloads\superfly\outputs\run_80_13DPE_n002\detections_raw.csv
Detection cache found — skipping RF-DETR: C:\Users\emmav\Downloads\superfly\outputs\run_80_13DPE_n002\detections_raw.csv
Saved: ..\outputs\run_107\tracks_wide_format.csv  (frames=324, tracks=45)
Relink: no swaps accepted.
Saved relinked tracks: ..\outputs\run_107\tracks_relinked.csv  (7500 rows)
(324, 46)
Saved detections video: ..\outputs\run_107\2024-02-12_NEG-008_h_detections_RF-DETR.mp4


In [16]:
# Quick mid-pipeline check: are detections reaching the tracker?
# No compact IDs yet (stitching hasn't run), so no output report saved here.
run_diagnostics(
    tracker     = tracker,
    df_wide     = df_wide,
    df_relinked          = df_relinked,
    df_stitched = None,
    n_expected  = 42,
    fps         = 30,
    config      = _cfg,
)

  RUN CONFIGURATION
{
  "roboflow": {
    "model_id": "flies-123/2"
  },
  "tracker": {
    "detection_confidence_rfdetr": 0.4,
    "confidence": 0.55,
    "track_activation_threshold": 0.1,
    "lost_track_buffer": 400,
    "minimum_matching_threshold": 0.2,
    "minimum_consecutive_frames": 1,
    "min_area": 20,
    "asso_func": "diou",
    "brownian_pos_noise": 15,
    "aspect_weight": 0.05,
    "behavioral_weight": 0.05,
    "jump_factor": 2.0,
    "jump_iou_threshold": 0.05,
    "jump_inertia": 0.05
  },
  "stitching": {
    "stitching_mode": "per_vial",
    "stop_mode": "converge",
    "max_rounds": 10,
    "w_under": 15,
    "w_over": 2.0,
    "vial_count_cap": 7,
    "general_count_cap": 56,
    "fps": 30,
    "pause_threshold": 1.0,
    "edge_fraction": 0.1,
    "min_points_for_scale": 10,
    "expected_per_vial": 7,
    "short_track_frac": 0.1,
    "link_score_weights": {
      "extrap": 0.4,
      "direction": 0.3,
      "behavioral": 0.3
    },
    "direction_weights": {
 

## 5 — No stitching (passthrough)

OC-SORT track IDs are used directly. `stitched_id` is set equal to `orig_id`.
`wide_to_long` converts the wide CSV to long format as usual.

In [17]:
STITCHED_CSV = os.path.join(OUTPUT_PATH, "tracks_xy_stitched_long.csv")
LONG_CSV     = os.path.join(OUTPUT_PATH, "tracks_long_format.csv")

with open(ROI_JSON) as f:
    vial_rois = {k: tuple(v) for k, v in json.load(f).items()}

long_df = wide_to_long(pd.read_csv(WIDE_CSV), out_csv=LONG_CSV)

# No stitching: stitched_id == orig_id
stitched_df = long_df.copy()
stitched_df["stitched_id"] = stitched_df["orig_id"]
stitched_df.to_csv(STITCHED_CSV, index=False)

print(f"Track IDs (no stitching): {stitched_df['orig_id'].nunique()}")
save_run_params(OUTPUT_PATH, "stitching_output", {
    "stitched_csv": STITCHED_CSV,
    "stitched_ids": int(stitched_df["stitched_id"].nunique()),
    "original_ids": int(stitched_df["orig_id"].nunique()),
})

Track IDs (no stitching): 45


## 6 — Vial assignment + compact IDs

Assigns each point to a vial using the ROI JSON, then assigns compact sequential IDs
(left → right within each vial).

In [18]:
COMPACT_CSV = os.path.join(OUTPUT_PATH, "compact_tracks.csv")

df_compact = assign_vials_and_compact_ids(
    stitched_csv=STITCHED_CSV,
    roi_json=ROI_JSON,
    out_csv=COMPACT_CSV,
    fps=_s.get("fps", 30),
)

print(df_compact.shape)
save_run_params(OUTPUT_PATH, "compact", {"csv": COMPACT_CSV, "rows": int(df_compact.shape[0])})
df_compact.head()

(7530, 8)


,frame,orig_id,x,y,stitched_id,vial_id,compact_id,fps
0,0,id1,426.02,332.43,id1,vial4,27,30.0
1,1,id1,409.08,296.57,id1,vial4,27,30.0
2,2,id1,407.31,288.96,id1,vial4,27,30.0
3,3,id1,408.25,300.46,id1,vial4,27,30.0
4,4,id1,416.81,320.87,id1,vial4,27,30.0


In [19]:
df_wide = pd.read_csv(WIDE_CSV)
num_frames = int(df_wide["frame"].max()) + 1
stitching_objectives = compute_stitching_objectives(
    df_stitched       = stitched_df,
    vial_rois         = vial_rois,
    num_frames        = num_frames,
    expected_per_vial = _s.get("expected_per_vial", 7),
    short_frac        = _s.get("short_track_frac", 0.10),
)
print_stitching_objectives(stitching_objectives)
save_run_params(OUTPUT_PATH, "stitching_objectives", {k: float(v) for k, v in stitching_objectives.items()})

# Full diagnostics: all three stages with compact IDs after stitching.
# Saves metrics_report.md + two PNG plots to OUTPUT_PATH.
run_diagnostics(
    tracker              = tracker,
    df_wide              = df_wide,
    df_stitched          = stitched_df,
    df_compact           = df_compact,
    df_relinked          = df_relinked,
    n_expected           = _s.get("expected_per_vial", 7) * len(vial_rois),
    fps                  = _s.get("fps", 30),
    vial_rois            = vial_rois,
    config               = _cfg,
    output_dir           = OUTPUT_PATH,
    stitching_objectives = stitching_objectives,
)

  STITCHING QUALITY OBJECTIVES
  vial_count_error      : 7.0  (0 = perfect)
  per_id_coverage_loss  : 156.7  frames/fly
  short_track_count     : 7
  per_frame_id_variance : 11.723
  RUN CONFIGURATION
{
  "roboflow": {
    "model_id": "flies-123/2"
  },
  "tracker": {
    "detection_confidence_rfdetr": 0.4,
    "confidence": 0.55,
    "track_activation_threshold": 0.1,
    "lost_track_buffer": 400,
    "minimum_matching_threshold": 0.2,
    "minimum_consecutive_frames": 1,
    "min_area": 20,
    "asso_func": "diou",
    "brownian_pos_noise": 15,
    "aspect_weight": 0.05,
    "behavioral_weight": 0.05,
    "jump_factor": 2.0,
    "jump_iou_threshold": 0.05,
    "jump_inertia": 0.05
  },
  "stitching": {
    "stitching_mode": "per_vial",
    "stop_mode": "converge",
    "max_rounds": 10,
    "w_under": 15,
    "w_over": 2.0,
    "vial_count_cap": 7,
    "general_count_cap": 56,
    "fps": 30,
    "pause_threshold": 1.0,
    "edge_fraction": 0.1,
    "min_points_for_scale": 10,
    "exp

INFO | Chromium init'ed with kwargs {}
INFO | Found chromium path: C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe
INFO | Temp directory created: C:\Users\emmav\AppData\Local\Temp\tmp9imj0rxx.
INFO | Opening browser.
INFO | Temp directory created: C:\Users\emmav\AppData\Local\Temp\tmpc4jffew6.
INFO | Temporary directory at: C:\Users\emmav\AppData\Local\Temp\tmpc4jffew6
INFO | Conforming 1 to file:///C:/Users/emmav/AppData/Local/Temp/tmp9imj0rxx/index.html
INFO | Waiting on all navigates
INFO | All navigates done, putting them all in queue.
INFO | Getting tab from queue (has 1)
INFO | Got FD1B
INFO | Processing XY_trajectories_Raw_tracker_IDs__Relinked_IDs__Compact_IDs_after_stitching.png
INFO | Sending big command for XY_trajectories_Raw_tracker_IDs__Relinked_IDs__Compact_IDs_after_stitching.png.
INFO | Sent big command for XY_trajectories_Raw_tracker_IDs__Relinked_IDs__Compact_IDs_after_stitching.png.
INFO | Reloading tab FD1B before return.
INFO | Putting tab FD1B back (

Report saved: ..\outputs\run_107\metrics_report.html
           +  ..\outputs\run_107\metrics_report.md


## 7 — Overlay video

Renders each fly as a coloured dot on the original video.

In [20]:
RAW_OVERLAY_MP4 = os.path.join(OUTPUT_PATH, f"{short_name}_overlay_raw_ocsort.mp4")
OVERLAY_MP4     = os.path.join(OUTPUT_PATH, f"{short_name}_overlay_vials_stitched.mp4")

# Pick the overlay substrate from config.yaml:visualization.overlay_source.
# Kept separate from PATH_TO_VID, which is the tracker input (_pp after Stage 2).
# In raw_cropped mode, prefer the saved cropped-raw clip when preprocessing ran.
# Fall back to RAW_VIDEO when no cropped-raw artifact exists.
_overlay_mode = _cfg.get("visualization", {}).get("overlay_source", "raw_cropped").lower()
if _overlay_mode == "raw_cropped" and RAW_CROPPED_VIDEO is not None:
    OVERLAY_VIDEO = str(RAW_CROPPED_VIDEO)
elif _overlay_mode == "raw_cropped":
    OVERLAY_VIDEO = RAW_VIDEO
else:
    OVERLAY_VIDEO = str(PATH_TO_VID)
print(f"overlay_source={_overlay_mode}  →  substrate: {OVERLAY_VIDEO}")

render_raw_overlay_video(
    video_path=OVERLAY_VIDEO,
    csv_path=LONG_CSV,
    out_mp4=RAW_OVERLAY_MP4,
    vial_rois=vial_rois,
    det_log_csv=DET_LOG_CSV,
)

render_vial_overlay_video(
    video_path=OVERLAY_VIDEO,
    csv_path=COMPACT_CSV,
    out_mp4=OVERLAY_MP4,
    vial_rois=vial_rois,
    det_log_csv=DET_LOG_CSV,
)

save_run_params(OUTPUT_PATH, "outputs", {"raw_overlay": RAW_OVERLAY_MP4, "overlay": OVERLAY_MP4})
Video(RAW_OVERLAY_MP4, width=800)

overlay_source=raw_cropped  →  substrate: ..\outputs\run_107\2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_002-converted_raw_cropped.mp4
Saved raw overlay video: ..\outputs\run_107\2024-02-12_NEG-008_h_overlay_raw_ocsort.mp4
Saved overlay video: ..\outputs\run_107\2024-02-12_NEG-008_h_overlay_vials_stitched.mp4
